## Extract

### Pokemon

In [2]:
import requests
url = "https://pokeapi.co/api/v2/pokemon/"

response = requests.get(url)
pokemon = response.json() 
pokemon['results']

[{'name': 'bulbasaur', 'url': 'https://pokeapi.co/api/v2/pokemon/1/'},
 {'name': 'ivysaur', 'url': 'https://pokeapi.co/api/v2/pokemon/2/'},
 {'name': 'venusaur', 'url': 'https://pokeapi.co/api/v2/pokemon/3/'},
 {'name': 'charmander', 'url': 'https://pokeapi.co/api/v2/pokemon/4/'},
 {'name': 'charmeleon', 'url': 'https://pokeapi.co/api/v2/pokemon/5/'},
 {'name': 'charizard', 'url': 'https://pokeapi.co/api/v2/pokemon/6/'},
 {'name': 'squirtle', 'url': 'https://pokeapi.co/api/v2/pokemon/7/'},
 {'name': 'wartortle', 'url': 'https://pokeapi.co/api/v2/pokemon/8/'},
 {'name': 'blastoise', 'url': 'https://pokeapi.co/api/v2/pokemon/9/'},
 {'name': 'caterpie', 'url': 'https://pokeapi.co/api/v2/pokemon/10/'},
 {'name': 'metapod', 'url': 'https://pokeapi.co/api/v2/pokemon/11/'},
 {'name': 'butterfree', 'url': 'https://pokeapi.co/api/v2/pokemon/12/'},
 {'name': 'weedle', 'url': 'https://pokeapi.co/api/v2/pokemon/13/'},
 {'name': 'kakuna', 'url': 'https://pokeapi.co/api/v2/pokemon/14/'},
 {'name': '

In [3]:
pokemon

{'count': 1351,
 'next': 'https://pokeapi.co/api/v2/pokemon/?offset=20&limit=20',
 'previous': None,
 'results': [{'name': 'bulbasaur',
   'url': 'https://pokeapi.co/api/v2/pokemon/1/'},
  {'name': 'ivysaur', 'url': 'https://pokeapi.co/api/v2/pokemon/2/'},
  {'name': 'venusaur', 'url': 'https://pokeapi.co/api/v2/pokemon/3/'},
  {'name': 'charmander', 'url': 'https://pokeapi.co/api/v2/pokemon/4/'},
  {'name': 'charmeleon', 'url': 'https://pokeapi.co/api/v2/pokemon/5/'},
  {'name': 'charizard', 'url': 'https://pokeapi.co/api/v2/pokemon/6/'},
  {'name': 'squirtle', 'url': 'https://pokeapi.co/api/v2/pokemon/7/'},
  {'name': 'wartortle', 'url': 'https://pokeapi.co/api/v2/pokemon/8/'},
  {'name': 'blastoise', 'url': 'https://pokeapi.co/api/v2/pokemon/9/'},
  {'name': 'caterpie', 'url': 'https://pokeapi.co/api/v2/pokemon/10/'},
  {'name': 'metapod', 'url': 'https://pokeapi.co/api/v2/pokemon/11/'},
  {'name': 'butterfree', 'url': 'https://pokeapi.co/api/v2/pokemon/12/'},
  {'name': 'weedle', '

### Weather

In [4]:
import csv
import gzip
from io import StringIO

import boto3
from botocore import UNSIGNED
from botocore.client import Config

bucket_name = "noaa-ghcn-pds"
file_key = "csv.gz/by_station/ASN00002022.csv.gz"

s3_client = boto3.client("s3", config=Config(signature_version=UNSIGNED))

response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
compressed_data = response["Body"].read()

csv_data = gzip.decompress(compressed_data).decode("utf-8")

csv_reader = csv.reader(StringIO(csv_data))
weather_data = list(csv_reader)
weather_data[:3]

[['ASN00002022', '20001130', 'PRCP', '0', '', '', 'a', ''],
 ['ASN00002022', '20001201', 'PRCP', '312', '', '', 'a', ''],
 ['ASN00002022', '20001202', 'PRCP', '0', '', '', 'a', '']]

### Orders & Customer

In [5]:
from sqlframe import activate
activate("duckdb")

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
customer_df = spark.read.csv("../data/customer.csv", header=True, inferSchema=True)

customer_df.show(2)

+----+------------+-----------+---------------+------------------------+--------------+------------+
| id | first_name | last_name | date_of_birth |         email          |     city     | state_code |
+----+------------+-----------+---------------+------------------------+--------------+------------+
| 1  |   James    |   Smith   |   1968-07-14  | james.smith@email.com  | Philadelphia |     PA     |
| 2  |    Mary    |  Johnson  |   2013-05-25  | mary.johnson@email.com | San Antonio  |     TX     |
+----+------------+-----------+---------------+------------------------+--------------+------------+


In [6]:
orders_df = spark.read.csv("../data/orders.csv", header=True, inferSchema=True)
orders_df.show(2)

+----------+-------------+---------+----------+--------+------------+
| order_id | customer_id | product | quantity | price  | order_date |
+----------+-------------+---------+----------+--------+------------+
|    1     |      20     |  Chair  |    3     | 732.14 | 2025-09-26 |
|    2     |      19     |  Mouse  |    1     | 516.8  | 2025-11-08 |
+----------+-------------+---------+----------+--------+------------+


## Transform 

### Weather

In [7]:
weather_header = ['id', 'date', 'element', 'value', 'm_flag', 'q_flag', 's_flag', 'obs_time']
weather_df = spark.createDataFrame(weather_data[1:], schema=weather_header)
weather_df.show(2)

+-------------+----------+---------+-------+--------+--------+--------+----------+
|      id     |   date   | element | value | m_flag | q_flag | s_flag | obs_time |
+-------------+----------+---------+-------+--------+--------+--------+----------+
| ASN00002022 | 20001201 |   PRCP  |  312  |        |        |   a    |          |
| ASN00002022 | 20001202 |   PRCP  |   0   |        |        |   a    |          |
+-------------+----------+---------+-------+--------+--------+--------+----------+


### Orders & Customer

In [8]:
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# 1. Left join orders -> customer (keep all orders, attach their customers)
#    orders.customer_id links to customer.id
# ---------------------------------------------------------------------------
enriched_orders_df = orders_df.join(
    customer_df,
    orders_df["customer_id"] == customer_df["id"],
    how="left",
)

# ---------------------------------------------------------------------------
# 2. Age from date_of_birth vs now (full years)
# ---------------------------------------------------------------------------
enriched_orders_df = enriched_orders_df.withColumn(
    "age",
    (F.year(F.current_date()) - F.year(F.to_date("date_of_birth"))).cast("int"),
)
enriched_orders_df.show(2)

+----------+-------------+------------+----------+-------+------------+----+------------+-----------+---------------+-------------------------+-------------+------------+-----+
| order_id | customer_id |  product   | quantity | price | order_date | id | first_name | last_name | date_of_birth |          email          |     city    | state_code | age |
+----------+-------------+------------+----------+-------+------------+----+------------+-----------+---------------+-------------------------+-------------+------------+-----+
|    97    |      2      | Headphones |    2     | 93.63 | 2025-06-03 | 2  |    Mary    |  Johnson  |   2013-05-25  |  mary.johnson@email.com | San Antonio |     TX     |  13 |
|   100    |      3      |    Desk    |    5     | 147.9 | 2025-09-20 | 3  |    John    |  Williams |   1961-07-01  | john.williams@email.com |   New York  |     NY     |  65 |
+----------+-------------+------------+----------+-------+------------+----+------------+-----------+--------------

In [9]:
# ---------------------------------------------------------------------------
# Age bucket: lt18, 18-30, 31-50, 51-70, 70+
# ---------------------------------------------------------------------------
enriched_orders_df = enriched_orders_df.withColumn(
    "age_bucket",
    F.when(F.col("age") < 18, "lt18")
     .when(F.col("age") <= 30, "18-30")
     .when(F.col("age") <= 50, "31-50")
     .when(F.col("age") <= 70, "51-70")
     .otherwise("70+"),
)
enriched_orders_df.show(2)

+----------+-------------+------------+----------+-------+------------+----+------------+-----------+---------------+-------------------------+-------------+------------+-----+------------+
| order_id | customer_id |  product   | quantity | price | order_date | id | first_name | last_name | date_of_birth |          email          |     city    | state_code | age | age_bucket |
+----------+-------------+------------+----------+-------+------------+----+------------+-----------+---------------+-------------------------+-------------+------------+-----+------------+
|    97    |      2      | Headphones |    2     | 93.63 | 2025-06-03 | 2  |    Mary    |  Johnson  |   2013-05-25  |  mary.johnson@email.com | San Antonio |     TX     |  13 |    lt18    |
|   100    |      3      |    Desk    |    5     | 147.9 | 2025-09-20 | 3  |    John    |  Williams |   1961-07-01  | john.williams@email.com |   New York  |     NY     |  65 |   51-70    |
+----------+-------------+------------+----------+

### Pokemon

In [10]:
import requests
 
# ---------------------------------------------------------------------------
# 1. Get 151 pokemon by following the "next" pagination link
# ---------------------------------------------------------------------------
url = "https://pokeapi.co/api/v2/pokemon/"
results = []
while url and len(results) < 151:
    resp = requests.get(url).json()
    results.extend(resp["results"])
    url = resp["next"]  # follow pagination
results = results[:151]

len(results)

151

In [11]:
# Better approach 
# Read the docs
url = "https://pokeapi.co/api/v2/pokemon/?limit=151"
results = requests.get(url).json()["results"]
len(results)

151

In [12]:
results[:2]

[{'name': 'bulbasaur', 'url': 'https://pokeapi.co/api/v2/pokemon/1/'},
 {'name': 'ivysaur', 'url': 'https://pokeapi.co/api/v2/pokemon/2/'}]

In [13]:
url = 'https://pokeapi.co/api/v2/pokemon/59/'
arcanine_data = requests.get(url).json()
print(f"id: {arcanine_data['id']}, name: {arcanine_data['name']}, base_experience: {arcanine_data['base_experience']}, height: {arcanine_data['height']}, weight: {arcanine_data['weight']}")

id: 59, name: arcanine, base_experience: 194, height: 19, weight: 1550


In [14]:
# ---------------------------------------------------------------------------
# For each pokemon, hit its url and get id, base_experience, height, weight
# Flatten into a list of dicts (one dict per pokemon)
# ---------------------------------------------------------------------------
pokemon_data = []
i = 0
for p in results:
    print(f'Hitting {p["url"]}')
    detail = requests.get(p["url"]).json()
    pokemon_data.append({
        "id": detail["id"],
        "name": detail["name"],
        "base_experience": detail["base_experience"],
        "height": detail["height"],
        "weight": detail["weight"],
    })
    i += 1
    if i > 5:
        break
# Slow and inefficient
# Read the pokeapi docs, look for graphql to get the data effectively (use LLM)

Hitting https://pokeapi.co/api/v2/pokemon/1/
Hitting https://pokeapi.co/api/v2/pokemon/2/
Hitting https://pokeapi.co/api/v2/pokemon/3/
Hitting https://pokeapi.co/api/v2/pokemon/4/
Hitting https://pokeapi.co/api/v2/pokemon/5/
Hitting https://pokeapi.co/api/v2/pokemon/6/


In [15]:
query = """
query {
  pokemon_v2_pokemon(where: {id: {_lte: 151}}, order_by: {id: asc}) {
    id
    base_experience
    height
    weight
    name
  }
}
"""

resp = requests.post(
    "https://beta.pokeapi.co/graphql/v1beta",
    json={"query": query},
)
pokemon_data = resp.json()["data"]["pokemon_v2_pokemon"]
print(len(pokemon_data), pokemon_data[58])

151 {'id': 59, 'base_experience': 194, 'height': 19, 'weight': 1550, 'name': 'arcanine'}


In [16]:
pokemon_df = spark.createDataFrame(pokemon_data)
pokemon_df.show(3)

+----+-----------------+--------+--------+-----------+
| id | base_experience | height | weight |    name   |
+----+-----------------+--------+--------+-----------+
| 1  |        64       |   7    |   69   | bulbasaur |
| 2  |       142       |   10   |  130   |  ivysaur  |
| 3  |       236       |   20   |  1000  |  venusaur |
+----+-----------------+--------+--------+-----------+


## Load

### Weather

### Enriched orders

### Pokemon

In [17]:
import duckdb

con = duckdb.connect("my_database.duckdb")

# Convert Spark DataFrames to pandas, then register + persist as DuckDB tables.
# CREATE OR REPLACE overwrites the table if it already exists.
for name, sdf in [
    ("pokemon", pokemon_df),
    ("weather", weather_df),
    ("enriched_orders", enriched_orders_df),
]:
    pdf = sdf.toPandas()
    con.register(f"{name}_tmp", pdf)
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM {name}_tmp")
    con.unregister(f"{name}_tmp")

con.close()

/home/josephkevinmachado/code/python_essentials_for_data_engineers/.venv/lib/python3.14/site-packages/sqlframe/base/session.py:588: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return read_sql_query(
/home/josephkevinmachado/code/python_essentials_for_data_engineers/.venv/lib/python3.14/site-packages/sqlframe/base/session.py:588: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return read_sql_query(
/home/josephkevinmachado/code/python_essentials_for_data_engineers/.venv/lib/python3.14/site-packages/sqlframe/base/session.py:588: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects a

In [18]:
# validate that the data was loaded 
con = duckdb.connect("my_database.duckdb")
for name in ["pokemon", "weather", "enriched_orders"]:
    print(f"\n=== {name} ===")
    print(con.execute(f"SELECT * FROM {name} LIMIT 2").fetchdf())
con.close()


=== pokemon ===
   id  base_experience  height  weight       name
0   1               64       7      69  bulbasaur
1   2              142      10     130    ivysaur

=== weather ===
            id      date element value m_flag q_flag s_flag obs_time
0  ASN00002022  20001201    PRCP   312                    a         
1  ASN00002022  20001202    PRCP     0                    a         

=== enriched_orders ===
   order_id  customer_id     product  quantity   price order_date   id  \
0        97            2  Headphones         2   93.63 2025-06-03  2.0   
1       100            3        Desk         5  147.90 2025-09-20  3.0   

  first_name last_name date_of_birth                    email         city  \
0       Mary   Johnson    2013-05-25   mary.johnson@email.com  San Antonio   
1       John  Williams    1961-07-01  john.williams@email.com     New York   

  state_code   age age_bucket  
0         TX  13.0       lt18  
1         NY  65.0      51-70  


In [19]:
# stakeholder query 
# Duplicated data 
con = duckdb.connect("my_database.duckdb")
report = "select order_id, count(*) from enriched_orders group by 1 having count(*) > 1"
print(con.execute(report).fetchdf())
con.close()

   order_id  count_star()
0        29             2
1        55             2


## Data Quality Check

In [20]:
con = duckdb.connect("my_database.duckdb")
enriched_orders_query = "select * from enriched_orders"
enriched_orders_data = con.execute(enriched_orders_query).fetchdf()
con.close()

In [21]:
import pointblank as pb

validation = (
    pb.Validate(data=enriched_orders_data, tbl_name="enriched_orders_data", thresholds=pb.Thresholds(error=1, critical=2) )

    # 1. Duplicate order id -> order_id values must be unique
    .rows_distinct(columns_subset=["order_id"])

    # 2. Null customer_name -> must not be null
    .col_vals_not_null(columns="first_name")

    # 3. Null purchase amount -> must not be null
    .col_vals_not_null(columns="price")

    .interrogate()
)

# Report / results
validation.get_tabular_report().show()
print("All passed:", validation.all_passed())

Pointblank Validation 
 
 
 2026-07-07|17:19:27 Pandas enriched_orders_data WARNING — ERROR 1 CRITICAL 2 
 
 
 
 
 STEP 
 COLUMNS 
 VALUES 
 TBL 
 EVAL 
 UNITS 
 PASS 
 FAIL 
 W 
 E 
 C 
 EXT 
 
 
 
 
 #FF3300 
 1 
 
 
 <!--?xml version="1.0" encoding="UTF-8"?--><?xml version="1.0" encoding="UTF-8"?>
 
 rows_distinct 
 
 
 <path d="M56.712234,1 C59.1975153,1 61.4475153,2.00735931 63.076195,3.63603897 C64.7048747,5.26471863 65.712234,7.51471863 65.712234,10 L65.712234,10 L65.712234,65 L10.712234,65 C8.22695259,65 5.97695259,63.9926407 4.34827294,62.363961 C2.71959328,60.7352814 1.71223397,58.4852814 1.71223397,56 L1.71223397,56 L1.71223397,10 C1.71223397,7.51471863 2.71959328,5.26471863 4.34827294,3.63603897 C5.97695259,2.00735931 8.22695259,1 10.712234,1 L10.712234,1 Z" id="rectangle" stroke="#000000" stroke-width="2" fill="#FFFFFF"> 
 
 <path d="M3.66705619,6.62107508 C3.12510104,6.64066386 2.67455974,7.04767444 2.60273432,7.58527682 C2.52873228,8.1228792 2.85303526,8.6343627 3.37104848,8.79760239 C4.40489909,9.15455286 6.70113553,9.87063021 9.86580595,10.364702 L9.86580595,30.7369976 C6.65978137,31.2332458 4.37225104,31.964559 3.37104848,32.3215095 C2.78991555,32.5282797 2.48520173,33.1681784 2.69197196,33.7493114 C2.8987422,34.3304443 3.53864094,34.6351581 4.11977387,34.4283879 C5.54975217,33.9190808 10.2031678,32.4608072 16.5172734,32.4608072 C22.7943781,32.4608072 27.5500901,33.8907855 29.0540706,34.4109757 C29.6352036,34.6133926 30.2707495,34.304326 30.4731664,33.7231931 C30.6755833,33.1420601 30.3665167,32.5065142 29.7853838,32.3040973 C28.7493568,31.9449704 26.4313554,31.2441283 23.2383897,30.7544098 L23.2383897,10.3821143 C26.4444143,9.88804243 28.745004,9.16108259 29.7679716,8.79760239 C30.3491045,8.59083215 30.6538184,7.95093341 30.4470481,7.36980048 C30.2402779,6.78866755 29.6003791,6.48395373 29.0192462,6.69072396 C27.5587963,7.20873774 22.9162637,8.65830464 16.6391589,8.65830464 C10.3707603,8.65830464 5.61287188,7.21309051 4.10236166,6.69072396 C3.96306391,6.6384873 3.81506005,6.61454548 3.66705619,6.62107508 Z M12.0945699,10.6432975 C13.4940771,10.789125 15.0198226,10.8870686 16.6391589,10.8870686 C18.1997289,10.8870686 19.6623552,10.7978311 21.0096257,10.6607098 L21.0096257,30.4584021 C19.623178,30.316928 18.1191975,30.2320433 16.5172734,30.2320433 C14.9327615,30.2320433 13.4570763,30.3191044 12.0945699,30.4584021 L12.0945699,10.6432975 Z" id="gemini" stroke="#000000" stroke-width="2" fill="#000000" fill-rule="nonzero"> 
 
 
 
 
 
 
 
 
 rows_distinct() 
 
 
 
 order_id 
 — 
 
 
 
 
 <path d="M5.80375046,8.18194736 C3.77191832,8.18194736 2.11875046,9.83495328 2.11875046,11.8669474 C2.11875046,13.8989414 3.77191832,15.5519474 5.80375046,15.5519474 C7.8355826,15.5519474 9.48875046,13.8989414 9.48875046,11.8669474 C9.48875046,9.83495328 7.83552863,8.18194736 5.80375046,8.18194736 Z M5.80375046,14.814915 C4.17821997,14.814915 2.85578285,13.4924778 2.85578285,11.8669474 C2.85578285,10.2414169 4.17821997,8.91897975 5.80375046,8.91897975 C7.42928095,8.91897975 8.75171807,10.2414169 8.75171807,11.8669474 C8.75171807,13.4924778 7.42928095,14.814915 5.80375046,14.814915 Z" id="Shape" fill="#000000" fill-rule="nonzero"> 
 <path d="M13.9638189,8.699335 C13.9364621,8.70430925 13.9091059,8.71176968 13.8842359,8.71923074 C13.7822704,8.73663967 13.6877654,8.77643115 13.6056956,8.83860518 L10.2433156,11.3852598 C10.0766886,11.5046343 9.97720993,11.6986181 9.97720993,11.9025491 C9.97720993,12.1064807 10.0766886,12.3004639 10.2433156,12.4198383 L13.6056956,14.966493 C13.891697,15.1803725 14.2970729,15.1231721 14.5109517,14.8371707 C14.7248313,14.5511692 14.6676309,14.145794 14.3816294,13.9319145 L12.5313257,12.5392127 L21.8812495,12.5392127 L21.8812495,11.2658854 L12.5313257,11.2658854 L14.3816294,9.87318364 C14.6377872,9.71650453 14.7497006,9.40066014 14.6477351,9.11714553 C14.5482564,8.83363156 14.262255,8.65954352 13.9638189,8.699335 Z" id="arrow" fill="#000000" transform="translate(15.929230, 11.894737) rotate(-180.000000) tr

All passed: False


In [22]:
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# 1. Left join orders -> customer (keep all orders, attach their customers)
#    orders.customer_id links to customer.id
# ---------------------------------------------------------------------------
enriched_orders_df = orders_df.join(
    customer_df,
    orders_df["customer_id"] == customer_df["id"],
    how="left",
)

# ---------------------------------------------------------------------------
# 2. Age from date_of_birth vs now (full years)
# ---------------------------------------------------------------------------
enriched_orders_df = enriched_orders_df.withColumn(
    "age",
    (F.year(F.current_date()) - F.year(F.to_date("date_of_birth"))).cast("int"),
)
enriched_orders_df.show(2)

# ---------------------------------------------------------------------------
# Age bucket: lt18, 18-30, 31-50, 51-70, 70+
# ---------------------------------------------------------------------------
enriched_orders_df = enriched_orders_df.withColumn(
    "age_bucket",
    F.when(F.col("age") < 18, "lt18")
     .when(F.col("age") <= 30, "18-30")
     .when(F.col("age") <= 50, "31-50")
     .when(F.col("age") <= 70, "51-70")
     .otherwise("70+"),
)
enriched_orders_df.show(2)

+----------+-------------+------------+----------+-------+------------+----+------------+-----------+---------------+-------------------------+-------------+------------+-----+
| order_id | customer_id |  product   | quantity | price | order_date | id | first_name | last_name | date_of_birth |          email          |     city    | state_code | age |
+----------+-------------+------------+----------+-------+------------+----+------------+-----------+---------------+-------------------------+-------------+------------+-----+
|    97    |      2      | Headphones |    2     | 93.63 | 2025-06-03 | 2  |    Mary    |  Johnson  |   2013-05-25  |  mary.johnson@email.com | San Antonio |     TX     |  13 |
|   100    |      3      |    Desk    |    5     | 147.9 | 2025-09-20 | 3  |    John    |  Williams |   1961-07-01  | john.williams@email.com |   New York  |     NY     |  65 |
+----------+-------------+------------+----------+-------+------------+----+------------+-----------+--------------

In [23]:
cleaned_enriched_orders_df = enriched_orders_df.dropDuplicates(["id"]).fillna({"first_name": "UNKNOWN", "price": 0})

import pointblank as pb

validation = (
    pb.Validate(data=cleaned_enriched_orders_df.toPandas(), tbl_name="cleaned_enriched_orders_df", thresholds=pb.Thresholds(error=1, critical=2) )

    # 1. Duplicate order id -> order_id values must be unique
    .rows_distinct(columns_subset=["order_id"])

    # 2. Null customer_name -> must not be null
    .col_vals_not_null(columns="first_name")

    # 3. Null purchase amount -> must not be null
    .col_vals_not_null(columns="price")

    .interrogate()
)

# Report / results
validation.get_tabular_report().show()
print("All passed:", validation.all_passed())

/home/josephkevinmachado/code/python_essentials_for_data_engineers/.venv/lib/python3.14/site-packages/sqlframe/base/session.py:588: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return read_sql_query(


Pointblank Validation 
 
 
 2026-07-07|17:19:27 Pandas cleaned_enriched_orders_df WARNING — ERROR 1 CRITICAL 2 
 
 
 
 
 STEP 
 COLUMNS 
 VALUES 
 TBL 
 EVAL 
 UNITS 
 PASS 
 FAIL 
 W 
 E 
 C 
 EXT 
 
 
 
 
 #4CA64C 
 1 
 
 
 <!--?xml version="1.0" encoding="UTF-8"?--><?xml version="1.0" encoding="UTF-8"?>
 
 rows_distinct 
 
 
 <path d="M56.712234,1 C59.1975153,1 61.4475153,2.00735931 63.076195,3.63603897 C64.7048747,5.26471863 65.712234,7.51471863 65.712234,10 L65.712234,10 L65.712234,65 L10.712234,65 C8.22695259,65 5.97695259,63.9926407 4.34827294,62.363961 C2.71959328,60.7352814 1.71223397,58.4852814 1.71223397,56 L1.71223397,56 L1.71223397,10 C1.71223397,7.51471863 2.71959328,5.26471863 4.34827294,3.63603897 C5.97695259,2.00735931 8.22695259,1 10.712234,1 L10.712234,1 Z" id="rectangle" stroke="#000000" stroke-width="2" fill="#FFFFFF"> 
 
 <path d="M3.66705619,6.62107508 C3.12510104,6.64066386 2.67455974,7.04767444 2.60273432,7.58527682 C2.52873228,8.1228792 2.85303526,8.6343627 3.37104848,8.79760239 C4.40489909,9.15455286 6.70113553,9.87063021 9.86580595,10.364702 L9.86580595,30.7369976 C6.65978137,31.2332458 4.37225104,31.964559 3.37104848,32.3215095 C2.78991555,32.5282797 2.48520173,33.1681784 2.69197196,33.7493114 C2.8987422,34.3304443 3.53864094,34.6351581 4.11977387,34.4283879 C5.54975217,33.9190808 10.2031678,32.4608072 16.5172734,32.4608072 C22.7943781,32.4608072 27.5500901,33.8907855 29.0540706,34.4109757 C29.6352036,34.6133926 30.2707495,34.304326 30.4731664,33.7231931 C30.6755833,33.1420601 30.3665167,32.5065142 29.7853838,32.3040973 C28.7493568,31.9449704 26.4313554,31.2441283 23.2383897,30.7544098 L23.2383897,10.3821143 C26.4444143,9.88804243 28.745004,9.16108259 29.7679716,8.79760239 C30.3491045,8.59083215 30.6538184,7.95093341 30.4470481,7.36980048 C30.2402779,6.78866755 29.6003791,6.48395373 29.0192462,6.69072396 C27.5587963,7.20873774 22.9162637,8.65830464 16.6391589,8.65830464 C10.3707603,8.65830464 5.61287188,7.21309051 4.10236166,6.69072396 C3.96306391,6.6384873 3.81506005,6.61454548 3.66705619,6.62107508 Z M12.0945699,10.6432975 C13.4940771,10.789125 15.0198226,10.8870686 16.6391589,10.8870686 C18.1997289,10.8870686 19.6623552,10.7978311 21.0096257,10.6607098 L21.0096257,30.4584021 C19.623178,30.316928 18.1191975,30.2320433 16.5172734,30.2320433 C14.9327615,30.2320433 13.4570763,30.3191044 12.0945699,30.4584021 L12.0945699,10.6432975 Z" id="gemini" stroke="#000000" stroke-width="2" fill="#000000" fill-rule="nonzero"> 
 
 
 
 
 
 
 
 
 rows_distinct() 
 
 
 
 order_id 
 — 
 
 
 
 
 <path d="M5.80375046,8.18194736 C3.77191832,8.18194736 2.11875046,9.83495328 2.11875046,11.8669474 C2.11875046,13.8989414 3.77191832,15.5519474 5.80375046,15.5519474 C7.8355826,15.5519474 9.48875046,13.8989414 9.48875046,11.8669474 C9.48875046,9.83495328 7.83552863,8.18194736 5.80375046,8.18194736 Z M5.80375046,14.814915 C4.17821997,14.814915 2.85578285,13.4924778 2.85578285,11.8669474 C2.85578285,10.2414169 4.17821997,8.91897975 5.80375046,8.91897975 C7.42928095,8.91897975 8.75171807,10.2414169 8.75171807,11.8669474 C8.75171807,13.4924778 7.42928095,14.814915 5.80375046,14.814915 Z" id="Shape" fill="#000000" fill-rule="nonzero"> 
 <path d="M13.9638189,8.699335 C13.9364621,8.70430925 13.9091059,8.71176968 13.8842359,8.71923074 C13.7822704,8.73663967 13.6877654,8.77643115 13.6056956,8.83860518 L10.2433156,11.3852598 C10.0766886,11.5046343 9.97720993,11.6986181 9.97720993,11.9025491 C9.97720993,12.1064807 10.0766886,12.3004639 10.2433156,12.4198383 L13.6056956,14.966493 C13.891697,15.1803725 14.2970729,15.1231721 14.5109517,14.8371707 C14.7248313,14.5511692 14.6676309,14.145794 14.3816294,13.9319145 L12.5313257,12.5392127 L21.8812495,12.5392127 L21.8812495,11.2658854 L12.5313257,11.2658854 L14.3816294,9.87318364 C14.6377872,9.71650453 14.7497006,9.40066014 14.6477351,9.11714553 C14.5482564,8.83363156 14.262255,8.65954352 13.9638189,8.699335 Z" id="arrow" fill="#000000" transform="translate(15.929230, 11.894737) rotate(-180.0000

All passed: True


In [24]:
import duckdb

con = duckdb.connect("my_database.duckdb")

# Convert Spark DataFrames to pandas, then register + persist as DuckDB tables.
# CREATE OR REPLACE overwrites the table if it already exists.
for name, sdf in [
    ("enriched_orders", cleaned_enriched_orders_df),
]:
    pdf = sdf.toPandas()
    con.register(f"{name}_tmp", pdf)
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM {name}_tmp")
    con.unregister(f"{name}_tmp")

con.close()

/home/josephkevinmachado/code/python_essentials_for_data_engineers/.venv/lib/python3.14/site-packages/sqlframe/base/session.py:588: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return read_sql_query(


In [25]:
# stakeholder query 
# Duplicated data 
con = duckdb.connect("my_database.duckdb")
report = "select order_id, count(*) from enriched_orders group by 1 having count(*) > 1"
print(con.execute(report).fetchdf())
con.close()

Empty DataFrame
Columns: [order_id, count_star()]
Index: []


## Code test

In [26]:
! uv run pytest ./

============================= test session starts ==============================
platform linux -- Python 3.14.5, pytest-9.1.1, pluggy-1.6.0
rootdir: /home/josephkevinmachado/code/python_essentials_for_data_engineers
configfile: pyproject.toml
plugins: pointblank-0.25.0, anyio-4.14.1
collected 1 item                                                               

test_transform.py .                                                      [100%]

============================== 1 passed in 0.63s ===============================
